# Memory & GPU · **Python side**

*Python / Colab track — analogue of `4_memory_and_gpu.jl`.*

Locality (numpy is **row-major**, the opposite of Julia), arithmetic intensity & **batching**, then
the **GPU with PyTorch**.

> For the GPU: Colab ▸ *Edit ▸ Notebook settings ▸ GPU*.

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Time per call (s), %timeit-style: auto-scale then min over `repeat` trials."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np

## 1. Locality: numpy is *row-major* (C-order)

Two elements of the same **row** are neighbours in memory (C-order) — **the opposite of Julia**
(column-major). We sum the same matrix along the same axis, but laid out in C order and then in
Fortran order: only the **layout** changes.

**Bet before you look:** the Julia twin of this experiment (a hand-written loop) measures a **12×**
gap between the two layouts. What will numpy show?

In [ ]:
N = 8000
A = np.random.rand(N, N)            # C-order (contiguous rows) by default
A_f = np.asfortranarray(A)          # same numbers, contiguous columns

# reduction along axis 0 (we walk down the columns):
t_c = best_time(lambda: A.sum(axis=0))     # C-order: columns are strided
t_f = best_time(lambda: A_f.sum(axis=0))   # F-order: columns are contiguous
print("sum axis=0, C-order :", ms(t_c))
print("sum axis=0, F-order :", ms(t_f))
print(f"ratio : {t_c/t_f:.2f}×")

**≈ 1×. The layout made no difference — and that is the lesson.**

numpy's reduction does not walk the axis in the order *you* wrote it: `nditer` inspects the strides
and **reorders the iteration into memory order**. Whatever axis you ask for, numpy reads the buffer
the fast way. It protected you.

A hand-written loop has no such luck — nobody reorders it. That is exactly why the Julia twin, which
writes the loop out by hand, measures **12×** where numpy measures 1×. The hardware penalty is
identical in both languages; numpy just hides it behind the library.

So to *see* the cost in Python, we need work numpy **cannot** reorder: a copy between two arrays
whose layouts disagree. The destination must be filled in C order while the source can only be read
in F order — someone has to pay for the transposition.

In [ ]:
outC = np.empty((N, N))                      # destination: C-order

t_same = best_time(lambda: np.copyto(outC, A))    # C ← C : layouts agree
t_diff = best_time(lambda: np.copyto(outC, A_f))  # C ← F : real transposition traffic
print("copyto  C ← C :", ms(t_same))
print("copyto  C ← F :", ms(t_diff))
print(f"ratio : {t_diff/t_same:.2f}×")

**≈ 14×**, in pure numpy, on the same bytes and the same number of elements. Only the *order* of
the memory accesses changed.

Two things to take away:

- Getting the direction wrong doesn't give a wrong result — just slower code. In numpy the
  contiguous axis by default is the **last** one (rows); in Julia it's the **first** (columns).
- **A library can hide the penalty; it cannot remove it.** numpy saved you on `sum(axis=0)` because
  someone wrote the reordering for you. The moment you step outside what the library can rearrange —
  a layout-mismatched copy here, a hand-written loop in Julia — the memory hierarchy sends the bill.

## 2. Arithmetic intensity: matrix-vector vs matrix-matrix

In [ ]:
n = 2000
M  = np.random.rand(n, n)
vv = np.random.rand(n)
M2 = np.random.rand(n, n)

print("matrix × vector :", ms(best_time(lambda: M @ vv)))   # ~2n² ops / n² data → memory-bound
print("matrix × matrix :", ms(best_time(lambda: M @ M2)))   # ~2n³ ops / n² data → compute-bound

- **mat × vector**: ~2 ops per datum loaded → we wait on memory (*memory-bound*).
- **mat × matrix**: ~n ops per datum → we saturate the compute (*compute-bound*, BLAS level 3).

## 3. Batching: throughput **per example** (numpy)

One "layer" `W` (n→n) applied to a batch of `B` examples stacked as columns (`W @ X`).

⚠️ We take **n = 4096, float32**, exactly like the Julia twin — and that size is not arbitrary:
`W` is then 4096² × 4 B = **64 MiB**, which can *never* sit in a typical **~24 MiB L3**. So every
batch has to **stream `W` from RAM**, and that is the fixed cost batching amortises. Shrink `W`
until it fits in cache and the whole effect goes away.

In [ ]:
nb = 4096                                    # same size as the Julia notebook
W = np.random.rand(nb, nb).astype(np.float32)   # 64 MiB — well past the L3
print("batch B | total (ms) | µs PER EXAMPLE | GFLOP/s")
print("-"*52)
for B in (1, 8, 64, 256, 1024):
    X = np.random.rand(nb, B).astype(np.float32)
    t = best_time(lambda: W @ X, repeat=3)
    print(f"{B:6d}  | {t*1e3:10.2f} | {t/B*1e6:14.2f} | {2.0*nb*nb*B/t/1e9:7.1f}")

The **total** time grows with `B` (we compute more), but the time **per example** *drops*: the same
`W`, streamed once, serves all `B` examples instead of one. That is "matrix-vector (memory-bound) →
matrix-matrix (compute-bound)", and the GFLOP/s column says it from the other side.

The curve **flattens** past B ≈ 256: `time/example ≈ compute/example + (cost of streaming W)/B`, so
once that fixed cost is amortised there is nothing left to win. Flattening is the **success signal**.

> **Batching is not a GPU trick** — it already pays here, on the CPU, no accelerator in sight.

## 4. GPU with PyTorch

`torch` picks CPU or CUDA from the *device* of the tensors — just as Julia's multiple dispatch
picks the GPU version for `CuArray`s. Same operations, different hardware.

In [ ]:
import torch
gpu = torch.cuda.is_available()
print("CUDA available :", gpu, "|", torch.cuda.get_device_name(0) if gpu else "CPU only")

def bench_torch(fn, gpu, repeat=5):
    if gpu: torch.cuda.synchronize()
    # warm-up
    fn();  torch.cuda.synchronize() if gpu else None
    import time
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter(); fn()
        if gpu: torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best

### A GPU tensor is **not** a CPU tensor

`.cuda()` **copies** the data across the PCIe bus into the card's memory and gives back a
*different object*, in a different address space. Two surprises worth meeting now rather than at
3 a.m.: the **dtype** (torch defaults to `float32`, numpy to `float64`), and what happens when you
mix devices.

*(Julia twin: `cu(A)` returns a `CuArray{Float32,...}` — same copy, same silent narrowing.)*

In [ ]:
a_np = np.random.rand(3, 3)          # numpy: float64
t_cpu = torch.rand(3, 3)             # torch: float32 by DEFAULT — not the same dtype!
print("numpy dtype :", a_np.dtype)
print("torch dtype :", t_cpu.dtype, "  <- float32, silently")
print("torch device:", t_cpu.device)

if gpu:
    t_gpu = t_cpu.cuda()             # THE COPY: host -> device
    print("after .cuda():", t_gpu.device, "|", type(t_gpu).__name__)
    print("same object? ", t_gpu is t_cpu, " | same values?", torch.equal(t_cpu, t_gpu.cpu()))
    # The classic beginner error: one operand on each side of the bus.
    try:
        t_cpu + t_gpu
    except RuntimeError as e:
        print("cpu_tensor + gpu_tensor ->", str(e).split(",")[0])
    # And .item() is a device -> host transfer in disguise:
    print("t_gpu.sum().item() =", round(t_gpu.sum().item(), 4), " <- this SYNCS and copies back")

### What the copy costs

The bus is not free. Time the trip **up**, the trip **back**, and the work in between.

In [ ]:
if gpu:
    K = 4096
    Ah = torch.rand(K, K)                                    # 64 MiB, float32, on the host
    Ad = Ah.cuda(); Bd = torch.rand(K, K, device="cuda")
    t_up   = bench_torch(lambda: Ah.cuda(),   gpu=True)      # host -> device
    t_down = bench_torch(lambda: Ad.cpu(),    gpu=True)      # device -> host
    t_mul  = bench_torch(lambda: Ad @ Bd,     gpu=True)      # the actual work
    print("host -> device :", ms(t_up))
    print("device -> host :", ms(t_down))
    print("matmul on GPU  :", ms(t_mul))
    print(f"-> the round trip costs {(t_up + t_down)/t_mul:.1f}x the computation it feeds")
else:
    print("(no GPU)")

**Rule: upload once, do many operations on the card, bring back only what you need** — ideally a
single scalar. Code that copies back after every step lives on the bus, not on the GPU. Note that
`pi_torch` below draws its random numbers **directly on the device** for exactly this reason, and
that its `.item()` is the *one* transfer it allows itself.

### Work in place

`z = 2*x + y` allocates a fresh output array **every call**. `np.add(..., out=z)` writes into
memory you already own. Same maths, no garbage, cache-hot buffer.

> **Watch the size of the win, and why it is smaller here than in Julia.** numpy **cannot fuse**:
> `2*x + y` builds a temporary for `2*x`, then adds — and the `out=` version still needs *two*
> passes over the data. Julia's `z .= 2 .* x .+ y` fuses the whole expression into **one** loop,
> which is why the Julia twin gains ~3× where numpy gains ~1.5×. Same idea, and the dot syntax is
> doing real work.

In [ ]:
m = 10_000_000
x = np.random.rand(m); y = np.random.rand(m); z = np.empty(m)

def out_of_place(): return 2*x + y             # allocates 80 MB every call
def in_place():                                 # writes into z, allocates nothing
    np.multiply(x, 2.0, out=z); np.add(z, y, out=z); return z

assert np.allclose(out_of_place(), in_place())
t_out = best_time(out_of_place, repeat=3)
t_in  = best_time(in_place,     repeat=3)
print("allocates :", ms(t_out))
print("in place  :", ms(t_in), f"-> {t_out/t_in:.1f}x, and 80 MB of garbage not created")

# On the GPU the same rule applies — an elementwise result this large is a real allocation:
if gpu:
    xg = torch.rand(m, device="cuda"); yg = torch.rand(m, device="cuda")
    zg = torch.empty(m, device="cuda")
    t_og = bench_torch(lambda: 2*xg + yg, gpu=True)
    t_ig = bench_torch(lambda: torch.add(yg, xg, alpha=2, out=zg), gpu=True)
    print("GPU allocates :", ms(t_og))
    print("GPU in place  :", ms(t_ig), f"-> {t_og/t_ig:.1f}x")

# ⚠️ Measure, don't assume. The win comes from AVOIDED MEMORY TRAFFIC, so it shrinks when the
#    output is small — and it can disappear entirely: the Julia twin finds NO gain from an
#    in-place `mul!` on the GPU, because there the result was tiny next to the operands.

In [ ]:
# Matrix product CPU vs GPU
M = 4096
Ac = torch.rand(M, M); Bc = torch.rand(M, M)
print("matmul CPU :", ms(bench_torch(lambda: Ac @ Bc, gpu=False)))
if gpu:
    Ag = Ac.cuda(); Bg = Bc.cuda()
    print("matmul GPU :", ms(bench_torch(lambda: Ag @ Bg, gpu=True)))

In [ ]:
# Batching CPU vs GPU: read the GPU TOTAL column, not just the speedup
if gpu:
    Wt = torch.rand(nb, nb, device="cuda")       # same 4096² as the numpy table above
    Wc = Wt.cpu()
    print("batch B | GPU total (ms) | GPU µs/ex | GPU GFLOP/s | CPU µs/ex | CPU/GPU")
    print("-"*74)
    for B in (1, 8, 64, 256, 1024):
        Xc = torch.rand(nb, B); Xg = Xc.cuda()
        tc = bench_torch(lambda: Wc @ Xc, gpu=False)
        tg = bench_torch(lambda: Wt @ Xg, gpu=True)
        print(f"{B:6d}  | {tg*1e3:14.3f} | {tg/B*1e6:9.2f} | "
              f"{2.0*nb*nb*B/tg/1e9:11.0f} | {tc/B*1e6:9.1f} | {tc/tg:6.1f}×")
else:
    print("(no GPU: enable the GPU accelerator in Colab)")

In [ ]:
# The red thread: Monte-Carlo π on GPU
def pi_torch(n, device):
    x = torch.rand(n, device=device); y = torch.rand(n, device=device)
    return 4.0 * torch.count_nonzero(x*x + y*y <= 1.0).item() / n

n_mc = 100_000_000
print("π (reference) :", np.pi)
print("π torch CPU :", pi_torch(n_mc, "cpu"))
if gpu:
    print("π torch GPU :", pi_torch(n_mc, "cuda"))
    print("CPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cpu"), gpu=False)))
    print("GPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cuda"), gpu=True)))

## Wrap-up — to compare with Julia/CUDA

| Idea | Python | Julia |
|---|---|---|
| memory order | **row-major** (C) | **column-major** |
| the accelerator's array | `torch.Tensor` on `cuda:0` (float32 default) | `CuArray{Float32}` (`cu` narrows) |
| moving data | `.cuda()` / `.cpu()` / `.item()` | `cu(...)` / `Array(...)` |
| batching | `W @ X` numpy/torch | `W * X` |
| in place | `np.add(..., out=z)` | `z .= ...` (**fused**), `mul!` |
| GPU | PyTorch (`.cuda()`) | CUDA.jl (`cu(...)`, dispatch) |

> Same story: respect **locality**, **batch** to move from memory-bound to compute-bound, **stop
> moving data** across the bus, and **feed** the GPU (only worth it at scale). Compare the GPU/CPU
> factor you get here with the one from the Julia notebook.